In [1]:
import os
import json
import pickle
import random
import numpy as np

In [2]:
import openai
openai.organization = os.environ['openai_organization'] 
openai.api_key = os.environ['openai_api_key']
#openai.Model.list()

# Load mapping

In [3]:
mapping = json.load(open("oct8/kgindex.json"))
entities = dict([(value, key) for key, value in mapping["e"].items()])
relations = \
    {0: 'is related to',
     1: 'has context of',
     2: 'is a',
     3: 'is synonym of',
     4: 'at location',
     5: 'is etymologically related to',
     6: 'is distinct from',
     7: 'has the last subevent',
     8: 'is used for',
     9: 'is similar to',
     10: 'desires',
     11: 'is antonym of',
     12: 'dbpedia',
     13: 'is a part of',
     14: 'is a form of',
     15: 'has a',
     16: 'is capable of',
     17: 'is an instance of',
     18: 'has a prerequisite of',
     19: 'is motivated by the goal of',
     20: 'is derived from',
     21: 'has subevent',
     22: 'causes',
     23: 'receives sanction by',
     24: 'has the property of',
     25: 'entails',
     26: 'has the first subevent',
     27: 'does not desire',
     28: 'causes desire',
     29: 'is made of',
     30: 'does not have the property of',
     31: 'is created by',
     32: 'is located near',
     33: 'is not capable of',
     34: 'is defined as',
     35: 'is a manner of'}

# Load Pickle files

In [4]:
path_profix = "oct8"

In [5]:
pickle_files = {}
for t in ["type0000",
          "type0001",
          "type0002",
          "type0004",
          "type0005",
          "type0008",
          "type0009",
          "type0007"]:
    path_to_pickle = path_profix + "/" + t + ".pickle"
    if os.path.isfile(path_to_pickle):
        if t not in pickle_files:
            pickle_files[t] = [path_to_pickle]
        else:
            pickle_files[t].append(path_to_pickle)
pickle_files

{'type0000': ['oct8/type0000.pickle'],
 'type0001': ['oct8/type0001.pickle'],
 'type0002': ['oct8/type0002.pickle'],
 'type0004': ['oct8/type0004.pickle'],
 'type0008': ['oct8/type0008.pickle']}

# Chatgpt Prompt

In [6]:
background_str ="""
[Goal]
Use your inherent knowledge to find the proper assignment of variables that mostly satisfies the given conditions.

[Background]
1. A variable is a placeholder that can be any concept in the world.
2. An assignment of a variable is to assign the concept to the variable.
3. Each condition involves two terms (concept or variable) and the relation between them.
4. Each condition has a [necessity value] that describes the necessary confidence. If the confidence value is greater than the necessity value, the condition is satisfied.
5. Each condition has an [importance value] that indicates the importance of satisfying this condition. 
6. Each assignment of variables will provide one or multiple evaluations for all conditions, by grounding the variables to the corresponding concepts. 
7. Each evaluation has a [confidence value] in the range of 0 to 1, where 0 means such evaluation is totally false, 1 means such evaluation is totally true. You can have your own judgment of [confidence value] based on your inherent knowledge about how confident such evaluation should be.
8. An evaluation is satisfied if its confidence value is greater than the necessity value that we provide.
9. For each satisfied evaluation, the assignment that causes this evaluation to gain a score, which is the result of the confidence value multiplied by the importance value.
10. In the end, you should compare the total scores of these provided candidate assignments to find the most suitable and proper candidate and output it to me. The most suitable and proper candidate is the candidate with the highest or maximal total score compared to other provided candidates in terms of assignments.


[Task input]
The input of this task contains three parts of declarations.
1. The statement of variables
2. The statement of conditions
3. The statement of the assignments to the variables

"""

format_sample_str = """
[1. The statement of variables]
We have the following variables
1. f1

[2. The statement of conditions]
1. f1 {relation1} {element1}; the necessity value is {alpha1}; the importance value is {beta1}. 
2. f1 {relation2} {element2}; the necessity value is {alpha2}; the importance value is {beta2}.

[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be only the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index only. It MUST look like:
"2
<new line>
<new line>"

"""

# Parser

# type0000

In [7]:
dataset_type0000 = pickle.load(open(pickle_files["type0000"][0], "rb"))
len(dataset_type0000)

254

In [14]:
def parser_single(formula):
    last = 0
    
    relation = ""
    element1 = ""
    element2 = ""
    alpha = 0
    beta = 0
    
    for i in range(len(formula)):
        if formula[i] == "(":
            relation = formula[0:i]
            last = i+1
            break
    
    for i in range(last, len(formula)):   
        if formula[i] == ",":
            element1 = formula[last:i]
            last = i+1
            break
            
    for i in range(last, len(formula)):   
        if formula[i] == ",":
            element2 = formula[last:i]
            last = i+1
            break
    
    for i in range(last, len(formula)):   
        if formula[i] == ",":
            alpha = float(formula[last:i-1]) / 100
            last = i+1
            break
    
    for i in range(last, len(formula)):   
        if formula[i] == ")":
            beta = float(formula[last:i])
            last = i+1
            break
    return relation, element1, element2, alpha, beta

format_type0000_str = """
[1. The statement of variables]
We have the following variables
1. f1

[2. The statement of conditions]
1.  {s1} {r1} f1; the necessity value is {alpha1}; the importance value is {beta1}. 

[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be only the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index only. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0000[:1]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    r1, s1, f1, alpha1, beta1 = parser_single(row[1])
    r1 = entities[row[2]["r1"]]
    s1 = entities[row[2]["s1"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0000_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{},".format(idx), "reply:{},".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))

pickle_file = open("oct8/type0000_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0, reply:1, mapped answer: 2417


In [15]:
outputs

[(0, 2417)]

# type0001

In [16]:
dataset_type0001 = pickle.load(open(pickle_files["type0001"][0], "rb"))
print(len(dataset_type0001))
dataset_type0001[:5]

390


[(0,
  '(r1(s1,e1,25%,0.6))&(r2(e1,f1,75%,0.6))',
  {'r1': 3, 'r2': 0, 's1': 13927},
  array([ 163, 2142, 2932, 1333]),
  array([0.9612, 0.9818, 1.0013, 1.0032]),
  1333),
 (1,
  '(r1(s1,e1,25%,0.6))&(r2(e1,f1,25%,0.6))',
  {'r1': 8, 'r2': 3, 's1': 2876},
  array([  13, 5681, 1978, 3264]),
  array([0.8512, 0.9612, 0.8512, 0.8512]),
  5681),
 (2,
  '(r1(s1,e1,75%,1.0))&(r2(e1,f1,25%,0.4))',
  {'r1': 0, 'r2': 0, 's1': 6763},
  array([12595,  9170,  5526,  1013]),
  array([0.9197, 0.993 , 1.0664, 1.1093]),
  1013),
 (3,
  '(r1(s1,e1,25%,0.6))&(r2(e1,f1,25%,0.5))',
  {'r1': 0, 'r2': 0, 's1': 1089},
  array([  259,  8792, 10383,  5898]),
  array([0.9165, 0.9241, 0.9256, 0.9413]),
  5898),
 (4,
  '(r1(s1,e1,25%,0.7))&(r2(e1,f1,75%,0.7))',
  {'r1': 0, 'r2': 2, 's1': 276},
  array([ 3858, 10483,  1328,  2914]),
  array([1.1214, 1.1822, 1.2036, 1.2573]),
  2914)]

In [17]:
def parser_type0001(formula):
    first_formula = formula[1:formula.index(")")+1]
    second_formula = formula[formula.index("&")+2:-1]
    
    r1, s1, e1, alpha1, beta1 = parser_single(first_formula)
    r2, e1, f1, alpha2, beta2 = parser_single(second_formula)
    
    return alpha1, beta1, alpha2, beta2


format_type0001_str = """
[1. The statement of variables]
We have the following variables
1. f1
2. e1

[2. The statement of conditions]
1.  {s1} {r1} e1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  e1 {r2} f1; the necessity value is {alpha2}; the importance value is {beta2}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be only the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index only. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0001[:1]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2 = parser_type0001(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    s1 = entities[row[2]["s1"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0001_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,alpha2=alpha2,beta2=beta2,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))
    
pickle_file = open("oct8/type0001_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0 reply: 4 mapped answer: 1333


In [18]:
outputs

[(0, 1333)]

# type0002

In [19]:
dataset_type0002 = pickle.load(open(pickle_files["type0002"][0], "rb"))
print(len(dataset_type0002))
dataset_type0002[:5]

23


[(0,
  '(r1(s1,f1,75%,0.4))&(r2(s2,f1,75%,0.8))',
  {'r1': 0, 'r2': 0, 's1': 1872, 's2': 5878},
  array([1952,   33, 2362, 8118]),
  array([0.8512, 0.9523, 0.8512, 0.8512]),
  33),
 (1,
  '(r1(s1,f1,25%,0.5))&(r2(s2,f1,25%,0.5))',
  {'r1': 8, 'r2': 8, 's1': 7699, 's2': 7699},
  array([ 497, 2351,  516, 1287]),
  array([0.7093, 1.    , 0.7093, 0.7093]),
  2351),
 (2,
  '(r1(s1,f1,25%,0.8))&(r2(s2,f1,75%,0.1))',
  {'r1': 0, 'r2': 0, 's1': 1583, 's2': 1583},
  array([ 730,  742, 1171, 1671]),
  array([0.6384, 0.8034, 0.6384, 0.6384]),
  742),
 (3,
  '(r1(s1,f1,25%,0.8))&(r2(s2,f1,25%,0.7))',
  {'r1': 3, 'r2': 3, 's1': 9075, 's2': 1290},
  array([1290, 4861, 3867, 6393]),
  array([1.064 , 1.1923, 1.064 , 1.064 ]),
  4861),
 (4,
  '(r1(s1,f1,75%,0.8))&(r2(s2,f1,25%,1.0))',
  {'r1': 0, 'r2': 0, 's1': 1258, 's2': 965},
  array([ 649, 4043, 1944, 2483]),
  array([1.1876, 1.2767, 1.3571, 1.3589]),
  2483)]

In [20]:
def parser_type0002(formula):
    return parser_type0001(formula)


format_type0002_str = """
[1. The statement of variables]
We have the following variables
1. f1

[2. The statement of conditions]
1.  {s1} {r1} f1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  {s2} {r2} f1; the necessity value is {alpha2}; the importance value is {beta2}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be only the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index only. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0002[:1]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2 = parser_type0002(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    s1 = entities[row[2]["s1"]]
    s2 = entities[row[2]["s2"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0002_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,s2=s2,alpha2=alpha2,beta2=beta2,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))

pickle_file = open("oct8/type0002_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0 reply: 1 mapped answer: 1952


In [21]:
outputs

[(0, 1952)]

# type0004

In [11]:
dataset_type0004 = pickle.load(open(pickle_files["type0004"][0], "rb"))
print(len(dataset_type0004))
dataset_type0004[:5]

357


[(0,
  '(r1(s1,f1,25%,0.6))&(r2(e1,f1,25%,0.5))',
  {'r1': 0, 'r2': 0, 's1': 3996},
  array([2717, 9557,  240, 6255]),
  array([0.8944, 0.9057, 0.9256, 0.965 ]),
  6255),
 (1,
  '(r1(s1,f1,25%,0.8))&(r2(e1,f1,25%,0.2))',
  {'r1': 6, 'r2': 0, 's1': 844},
  array([11199,   136,  8397,  3305]),
  array([0.4022, 0.534 , 0.6649, 0.7102]),
  3305),
 (2,
  '(r1(s1,f1,75%,0.8))&(r2(e1,f1,75%,0.6))',
  {'r1': 0, 'r2': 0, 's1': 7862},
  array([6804, 4440,  699, 7878]),
  array([1.0802, 1.0893, 1.1674, 1.2376]),
  7878),
 (3,
  '(r1(s1,f1,25%,1.0))&(r2(e1,f1,75%,0.3))',
  {'r1': 4, 'r2': 0, 's1': 386},
  array([6332, 1517, 6138, 4233]),
  array([0.951 , 1.1783, 1.2224, 1.2564]),
  4233),
 (4,
  '(r1(s1,f1,25%,0.3))&(r2(e1,f1,25%,0.7))',
  {'r1': 0, 'r2': 0, 's1': 629},
  array([1468,  595, 1953,  259]),
  array([0.93  , 0.9378, 0.995 , 1.    ]),
  259)]

In [24]:
def parser_type0004(formula):
    return parser_type0001(formula)


format_type0004_str = """
[1. The statement of variables]
We have the following variables
1. f1
2. e1

[2. The statement of conditions]
1.  {s1} {r1} f1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  e1 {r2} f1; the necessity value is {alpha2}; the importance value is {beta2}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be only the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index only. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0004[:1]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2 = parser_type0004(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    s1 = entities[row[2]["s1"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0004_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,      alpha2=alpha2,beta2=beta2,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))


pickle_file = open("oct8/type0004_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0 reply: 2 mapped answer: 9557


In [26]:
outputs

[(0, 9557)]

# type0005

In [28]:
dataset_type0005 = pickle.load(open(pickle_files["type0005"][0], "rb"))
print(len(dataset_type0005))
dataset_type0005[:5]

KeyError: 'type0005'

In [29]:
def parser_type0005(formula):
    f = formula.split("&")
    first_formula = f[0][1:-1]
    second_formula = f[1][2:-1]
    third_formula = f[2][1:-2]
    r1, s1, e1, alpha1, beta1 = parser_single(first_formula)
    r2, e1, f1, alpha2, beta2 = parser_single(second_formula)
    r3, e1, f1, alpha3, beta3 = parser_single(third_formula)
    
    return alpha1, beta1, alpha2, beta2, alpha3, beta3

In [ ]:
format_type0005_str = """
[1. The statement of variables]
We have the following variables
1. f1
2. e1

[2. The statement of conditions]
1.  {s1} {r1} e1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  e1 {r2} f1; the necessity value is {alpha2}; the importance value is {beta2}. 
3.  e1 {r3} f1; the necessity value is {alpha3}; the importance value is {beta3}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be ONLY the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index ONLY. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0005[:1]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2, alpha3, beta3 = parser_type0005(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    r3 = entities[row[2]["r3"]]
    
    s1 = entities[row[2]["s1"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0005_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,      alpha2=alpha2,beta2=beta2,
                                                               r3=r3,      alpha3=alpha3,beta3=beta3,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))

print(outputs)
pickle_file = open("oct8/type0005_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

# type0008

In [30]:
dataset_type0008 = pickle.load(open(pickle_files["type0008"][0], "rb"))
print(len(dataset_type0008))
dataset_type0008[:5]

316


[(0,
  '(r1(s1,e1,75%,0.5))&((r2(s2,e1,75%,0.4))&(r3(e1,f1,25%,0.7)))',
  {'r1': 0, 'r2': 0, 'r3': 3, 's1': 3822, 's2': 2002},
  array([4879,  132,  913, 2557]),
  array([1.0065, 1.1349, 1.2633, 1.1349]),
  913),
 (1,
  '(r1(s1,e1,25%,0.3))&((r2(s2,e1,75%,0.9))&(r3(e1,f1,25%,0.4)))',
  {'r1': 0, 'r2': 9, 'r3': 0, 's1': 6404, 's2': 1708},
  array([ 773, 1569, 1466, 6546]),
  array([1.2844, 1.2979, 1.3033, 1.3507]),
  6546),
 (2,
  '(r1(s1,e1,75%,0.2))&((r2(s2,e1,25%,0.5))&(r3(e1,f1,25%,0.6)))',
  {'r1': 0, 'r2': 0, 'r3': 0, 's1': 8131, 's2': 8131},
  array([5004, 2250, 3370, 2873]),
  array([1.0831, 1.0933, 1.0939, 1.0965]),
  2873),
 (3,
  '(r1(s1,e1,75%,0.4))&((r2(s2,e1,25%,0.9))&(r3(e1,f1,75%,0.2)))',
  {'r1': 0, 'r2': 0, 'r3': 0, 's1': 4196, 's2': 4196},
  array([13095,   101,  2366,   161]),
  array([1.117 , 1.1193, 1.1198, 1.1221]),
  161),
 (4,
  '(r1(s1,e1,25%,0.6))&((r2(s2,e1,75%,0.7))&(r3(e1,f1,75%,0.5)))',
  {'r1': 0, 'r2': 0, 'r3': 0, 's1': 5209, 's2': 5209},
  array([13645,

In [31]:
def parser_type0008(formula):
    return parser_type0005(formula)

format_type0008_str = """
[1. The statement of variables]
We have the following variables
1. f1
2. e1

[2. The statement of conditions]
1.  {s1} {r1} e1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  {s2} {r2} e1; the necessity value is {alpha2}; the importance value is {beta2}. 
3.  e1 {r3} f1; the necessity value is {alpha3}; the importance value is {beta3}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be ONLY the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index ONLY. It MUST look like:
"2
<new line>
<new line>"

"""


outputs = []
for row in dataset_type0008[:1]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2, alpha3, beta3 = parser_type0008(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    r3 = entities[row[2]["r3"]]
    
    s1 = entities[row[2]["s1"]]
    s2 = entities[row[2]["s2"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0008_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,s2=s2,alpha2=alpha2,beta2=beta2,
                                                               r3=r3,      alpha3=alpha3,beta3=beta3,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))
    
pickle_file = open("oct8/type0008_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0 reply: 1 mapped answer: 4879


In [32]:
print(outputs)

[(0, 4879)]


# type0009

In [33]:
dataset_type0009 = pickle.load(open(pickle_files["type0009"][0], "rb"))
print(len(dataset_type0009))
dataset_type0009[:5]

KeyError: 'type0009'

In [34]:
def parser_type0009(formula):
    f = formula.split("&")
    first_formula = f[0][1:-1]
    second_formula = f[1][2:-1]
    third_formula = f[2][2:-1]
    forth_formula = f[3][1:-3]
    r1, s1, e1, alpha1, beta1 = parser_single(first_formula)
    r2, s2, e1, alpha2, beta2 = parser_single(second_formula)
    r3, e1, f1, alpha3, beta3 = parser_single(third_formula)
    r4, e1, f1, alpha4, beta4 = parser_single(forth_formula)
    return alpha1, beta1, alpha2, beta2, alpha3, beta3, alpha4, beta4

In [35]:
format_type0009_str = """
[1. The statement of variables]
We have the following variables
1. f1
2. e1

[2. The statement of conditions]
1.  {s1} {r1} e1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  {s2} {r2} e1; the necessity value is {alpha2}; the importance value is {beta2}. 
3.  e1 {r3} f1; the necessity value is {alpha3}; the importance value is {beta3}. 
4.  e1 {r4} f1; the necessity value is {alpha4}; the importance value is {beta4}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be ONLY the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index ONLY. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0009[:1]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2, alpha3, beta3, alpha4, beta4 = parser_type0009(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    r3 = entities[row[2]["r3"]]
    r4 = entities[row[2]["r4"]]
    
    s1 = entities[row[2]["s1"]]
    s2 = entities[row[2]["s2"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0009_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,s2=s2,alpha2=alpha2,beta2=beta2,
                                                               r3=r3,      alpha3=alpha3,beta3=beta3,
                                                               r4=r4,      alpha4=alpha4,beta4=beta4,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))
    
pickle_file = open("oct8/type0009_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

NameError: name 'dataset_type0009' is not defined

In [36]:
outputs

[]